In [45]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

class DualSimplexMaster:
    def __init__(self, c, A, b, name="Задача"):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.name = name
        self.num_constraints, self.num_vars = self.A.shape
        
        self.var_names = [f'$x_{{{i+1}}}$' for i in range(self.num_vars + self.num_constraints)]
        self.basis = [self.num_vars + i for i in range(self.num_constraints)]
        
        self.full_c = np.concatenate([self.c, np.zeros(self.num_constraints)])
        self.full_A = np.hstack([self.A, np.eye(self.num_constraints)])

    def _smart_round(self, val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return "—"
        if isinstance(val, (int, float, np.float64)):
            if abs(val - round(val)) < 1e-7:
                return str(int(round(val)))
            return f"{val:.2f}"
        return val

    def _get_z_row(self):
        cb = self.full_c[self.basis]
        return np.dot(cb, self.full_A) - self.full_c

    def _render_step(self, iteration, method_name, pivot_row=None, pivot_col=None):
        cb = self.full_c[self.basis]
        z_val = np.dot(cb, self.b)
        z_coeffs = self._get_z_row()
        
        data = []
        # Настройка формулы в зависимости от фазы
        if method_name == "Dual":
            ratio_label = '$-\\frac{Z_j - C_j}{a_{lj}}$'
        elif method_name == "Primal":
            ratio_label = '$\\frac{B_i}{a_{ik}}$'
        else:
            ratio_label = 'Отношение'

        for i, b_idx in enumerate(self.basis):
            row_prefix = [self.var_names[b_idx], cb[i], self.b[i]]
            # Если прямой метод, считаем отношение для строк
            if method_name == "Primal" and pivot_col is not None:
                val = self.full_A[i, pivot_col]
                ratio_val = self.b[i] / val if val > 1e-7 else None
                row_prefix[0] = f"{self.var_names[b_idx]} | {self._smart_round(ratio_val)}"
            
            row = row_prefix + list(self.full_A[i])
            data.append([self._smart_round(x) for x in row])

        z_row = ['$Z_j - C_j$', "—", z_val] + list(z_coeffs)
        data.append([self._smart_round(x) for x in z_row])
        
        # Строка отношений для двойственного метода (по столбцам)
        ratio_row = [ratio_label, "—", "—"]
        if method_name == "Dual" and pivot_row is not None:
            for j in range(len(self.var_names)):
                val = self.full_A[pivot_row, j]
                ratio_row.append(-(z_coeffs[j] / val) if val < -1e-7 else None)
        else:
            ratio_row.extend([None] * len(self.var_names))
        
        data.append([self._smart_round(x) for x in ratio_row])

        top_row_raw = ["", "", ""] + [self._smart_round(x) for x in self.full_c]
        top_row_unique = [val + (" " * i) for i, val in enumerate(top_row_raw)]
        
        columns = pd.MultiIndex.from_tuples(zip(top_row_unique, ['$БП$', '$C$', '$B$'] + self.var_names))
        df = pd.DataFrame(data, columns=columns)

        styles = [
            {'selector': 'th', 'props': [('border', '1px solid black'), ('text-align', 'center'), ('background-color', '#fff'), ('padding', '8px')]},
            {'selector': 'td', 'props': [('border', '1px solid gray'), ('text-align', 'center'), ('min-width', '70px'), ('padding', '8px'), ('color', 'black')]},
            {'selector': f'tr:nth-child({len(self.basis)+1})', 'props': [('background-color', '#f8f9fa'), ('font-weight', 'bold')]},
            {'selector': f'tr:nth-child({len(self.basis)+2})', 'props': [('font-style', 'italic'), ('color', '#d63384')]}
        ]
        display(HTML(f"<b>Итерация {iteration}</b>"))
        display(df.style.hide(axis='index').set_table_styles(styles))

    def _pivot(self, row, col):
        pivot_element = self.full_A[row, col]
        self.full_A[row] /= pivot_element
        self.b[row] /= pivot_element
        for i in range(self.num_constraints):
            if i != row:
                factor = self.full_A[i, col]
                self.full_A[i] -= factor * self.full_A[row]
                self.b[i] -= factor * self.b[row]
        self.basis[row] = col

    def solve(self):
        display(HTML(f"<h2 style='color: #2c3e50; border-bottom: 2px solid #eee;'>{self.name}</h2>"))
        iteration = 0
        
        # --- ФАЗА 1: Dual ---
        while np.any(self.b < -1e-7):
            pivot_row = np.argmin(self.b)
            z_row = self._get_z_row()
            ratios = [-(z_row[j] / self.full_A[pivot_row, j]) if self.full_A[pivot_row, j] < -1e-7 else np.inf for j in range(len(z_row))]
            
            if np.all(np.array(ratios) == np.inf):
                display(HTML("<b style='color:red;'>Решение отсутствует</b>")); return
                
            pivot_col = np.argmin(ratios)
            self._render_step(iteration, "", pivot_row=pivot_row)
            self._pivot(pivot_row, pivot_col)
            iteration += 1

        # --- ФАЗА 2: Primal ---
        while True:
            z_row = self._get_z_row()
            if np.all(z_row >= -1e-7): break
                
            pivot_col = np.argmin(z_row)
            limits = [self.b[i] / self.full_A[i, pivot_col] if self.full_A[i, pivot_col] > 1e-7 else np.inf for i in range(self.num_constraints)]
            
            if np.all(np.array(limits) == np.inf):
                display(HTML("<b style='color:red;'>Область не ограничена</b>")); return
                
            pivot_row = np.argmin(limits)
            self._render_step(iteration, "", pivot_col=pivot_col)
            self._pivot(pivot_row, pivot_col)
            iteration += 1

        self._render_step(iteration, "")
        
        final_values = {f'x_{{{i+1}}}': 0.0 for i in range(self.num_vars)}
        for i, b_idx in enumerate(self.basis):
            if b_idx < self.num_vars: final_values[f'x_{{{b_idx+1}}}'] = self.b[i]
        
        coords = [self._smart_round(final_values[f'x_{{{i+1}}}']) for i in range(self.num_vars)]
        z_val = np.dot(self.full_c[self.basis], self.b)
        
        display(HTML(f"<div style='border: 2px solid #28a745; padding: 10px; margin-top: 10px;'><b>Ответ:</b> $X^* = ({', '.join(coords)})$, $Z_{{max}} = {self._smart_round(z_val)}$</div>"))

# Запуск
# DualSimplexMaster(c=[3, 3], A=[[3, 4], [4, -6], [-4, 3]], b=[13, 1, -4]).solve()



DualSimplexMaster(
    c=[3, 3],
    A=[
        [3, 4], 
        [4, -6], 
        [-4, 3]
    ], 
    b=[13, 1, -4],
    name="Решение"
).solve()


# 1. ЕДИНСТВЕННОЕ РЕШЕНИЕ
DualSimplexMaster(
    c=[-1, -2, -1], 
    A=[
        [1, 1, 0],    # x1 + x2 <= 2
        [-2, -1, -1],  # -2x1 - x2 - x3 <= -4
        [0, -1, -2]    # -x2 - 2x3 <= -3
    ], 
    b=[2, -4, -3], 
    name="1. Стандартное решение"
).solve()

# 2. НЕСОВМЕСТНОСТЬ
# В строке с отрицательным свободным членом B нет отрицательных коэффициентов a_ij.
# Это означает, что область допустимых решений пуста.
DualSimplexMaster(
    c=[-1, -1, -1], 
    A=[
        [1, 1, 1],    # x1 + x2 + x3 <= 1
        [-1, -1, -1]   # x1 + x2 + x3 >= 3
    ], 
    b=[1, -3], 
    name="2. Несовместность"
).solve()

# 3. ВЫРОЖДЕННОЕ РЕШЕНИЕ
# Случай, когда на одной из итераций или в финале свободный член B становится равным 0.
# Геометрически это означает, что более двух прямых ограничений пересекаются в одной точке.
DualSimplexMaster(
    c=[-2, -2, -2], 
    A=[
        [-1, -1, 0], 
        [0, -1, -1],
        [-1, 0, -1]
    ], 
    b=[-2, -2, -2], 
    name="3. Вырожденное решение"
).solve()

# 4. МНОЖЕСТВО РЕШЕНИЙ
# В финальной таблице оценка Zj - Cj для небазисной переменной равна 0.
# Это значит, что целевая функция параллельна одному из ограничений.
DualSimplexMaster(
    c=[-1, -1, 0], 
    A=[
        [-1, -1, 0],   # Параллельно целевой функции
        [1, 0, 1], 
        [0, -2, -1]
    ], 
    b=[-3, 4, -2], 
    name="4. Множество решений"
).solve()

,,,3,3,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$
$x_{3}$,0,13,3,4,1,0,0
$x_{4}$,0,1,4,-6,0,1,0
$x_{5}$,0,-4,-4,3,0,0,1
$Z_j - C_j$,—,0,-3,-3,0,0,0
Отношение,—,—,—,—,—,—,—


,,,3,3,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$
$x_{3}$,0,10,0,6.25,1,0,0.75
$x_{4}$,0,-3,0,-3,0,1,1
$x_{1}$,3,1,1,-0.75,0,0,-0.25
$Z_j - C_j$,—,3,0,-5.25,0,0,-0.75
Отношение,—,—,—,—,—,—,—


,,,3,3,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$
$x_{3}$,0,3.75,0,0,1,2.08,2.83
$x_{2}$,3,1,0,1,0,-0.33,-0.33
$x_{1}$,3,1.75,1,0,0,-0.25,-0.50
$Z_j - C_j$,—,8.25,0,0,0,-1.75,-2.50
Отношение,—,—,—,—,—,—,—


,,,3,3,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$
$x_{5}$,0,1.32,0,0,0.35,0.74,1
$x_{2}$,3,1.44,0,1,0.12,-0.09,0
$x_{1}$,3,2.41,1,0,0.18,0.12,0
$Z_j - C_j$,—,11.56,0,0,0.88,0.09,0
Отношение,—,—,—,—,—,—,—


,,,-1,-2,-1,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$,$x_{6}$
$x_{4}$,0,2,1,1,0,1,0,0
$x_{5}$,0,-4,-2,-1,-1,0,1,0
$x_{6}$,0,-3,0,-1,-2,0,0,1
$Z_j - C_j$,—,0,1,2,1,0,0,0
Отношение,—,—,—,—,—,—,—,—


...

,,,-2,-2,-2,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$,$x_{6}$
$x_{1}$,-2,0,1,0,-1,-1,1,0
$x_{2}$,-2,2,0,1,1,0,-1,0
$x_{6}$,0,-2,0,0,-2,-1,1,1
$Z_j - C_j$,—,-4,0,0,2,2,0,0
Отношение,—,—,—,—,—,—,—,—


,,,-2,-2,-2,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$,$x_{6}$
$x_{1}$,-2,1,1,0,0,-0.50,0.50,-0.50
$x_{2}$,-2,1,0,1,0,-0.50,-0.50,0.50
$x_{3}$,-2,1,0,0,1,0.50,-0.50,-0.50
$Z_j - C_j$,—,-6,0,0,0,1,1,1
Отношение,—,—,—,—,—,—,—,—


,,,-1,-1,0,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$,$x_{6}$
$x_{4}$,0,-3,-1,-1,0,1,0,0
$x_{5}$,0,4,1,0,1,0,1,0
$x_{6}$,0,-2,0,-2,-1,0,0,1
$Z_j - C_j$,—,0,1,1,0,0,0,0
Отношение,—,—,—,—,—,—,—,—


,,,-1,-1,0,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$,$x_{6}$
$x_{1}$,-1,3,1,1,0,-1,0,0
$x_{5}$,0,1,0,-1,1,1,1,0
$x_{6}$,0,-2,0,-2,-1,0,0,1
$Z_j - C_j$,—,-3,0,0,0,1,0,0
Отношение,—,—,—,—,—,—,—,—


,,,-1,-1,0,0,0,0
$БП$,$C$,$B$,$x_{1}$,$x_{2}$,$x_{3}$,$x_{4}$,$x_{5}$,$x_{6}$
$x_{1}$,-1,2,1,0,-0.50,-1,0,0.50
$x_{5}$,0,2,0,0,1.50,1,1,-0.50
$x_{2}$,-1,1,0,1,0.50,0,0,-0.50
$Z_j - C_j$,—,-3,0,0,0,1,0,0
Отношение,—,—,—,—,—,—,—,—
